<a href="https://www.kaggle.com/code/zainabs1/marketing-campaign-performance-funnel-analysis?scriptVersionId=298101241" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Overview
This project analyzes **marketing campaign performance** using event-level advertising data.

The objective is to evaluate funnel performance, campaign efficiency, and audience behavior to identify optimization opportunities.

The analysis focuses on practical KPI construction and structured performance comparison to support Business Intelligence decision-making.

In [1]:
!pip install duckdb

In [2]:
#import necessary libraries
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

## Database Understanding

In [3]:
#create the SQL engine
engine = create_engine("sqlite:////kaggle/input/social-media-advertisement-performance/ad_campaign_db.sqlite")

In [4]:
tables = pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table'
ORDER BY name ASC;
""", 
                       engine)
display(tables)

,name
0,ad_events
1,ads
2,campaigns
3,users


#### A. Row Counts & Tables Inspection

In [5]:
for table in tables['name']:
    #row counts of each table
    row_count = pd.read_sql(
        f"""
        SELECT COUNT(*) AS count 
        FROM {table}
        """, 
            engine)
    #table inspection
    inspect = pd.read_sql(
        f"""
        SELECT * 
        FROM {table} 
        LIMIT 5;
        """, 
            engine)
    
    print(f"\n📌 {table}")
    display(row_count)
    display(inspect)


📌 ad_events


,count
0,400000


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression



📌 ads


,count
0,200


...

,count
0,50


,campaign_id,name,start_date,end_date,duration_days,total_budget
0,1,Campaign_1_Launch,2025-05-25,2025-07-23,59,24021.32
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,82,79342.41
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,56,14343.25
3,4,Campaign_4_Summer,2025-06-04,2025-08-08,65,45326.60
4,5,Campaign_5_Launch,2025-07-11,2025-08-28,48,68376.69



📌 users


,count
0,10000


,user_id,user_gender,user_age,age_group,country,location,interests
0,a2474,Female,24,18-24,United Kingdom,New Mariomouth,"fitness, health"
1,141e5,Male,21,18-24,Germany,Danielsfort,"food, fitness, lifestyle"
2,34db0,Male,27,25-34,Australia,Vincentchester,"fashion, news"
3,20d08,Female,28,25-34,India,Lisaport,"health, news, finance"
4,9e830,Male,28,25-34,United States,Brownmouth,"health, photography, lifestyle"


#### B. Event Type Distribution

In [6]:
event_type = pd.read_sql(
    """
    SELECT
        event_type,
        COUNT(*) AS event_count
    FROM ad_events
    GROUP BY event_type
    ORDER BY event_count DESC;
    """, 
        engine)

display(event_type)

,event_type,event_count
0,Impression,339812
1,Click,40079
2,Like,12013
3,Comment,4108
4,Purchase,2031
5,Share,1957


This clear decrease from impressions → clicks → purchases confirms a realistic funnel structure.

## Funnel Analysis

#### 1. Funnel Construction
The funnel stages are defined as:

`Impression` → `Click` → `Purchase`

**Note**: Metrics are calculated using event-level aggregation.

In [7]:
#funnel stages
metrics1 = pd.read_sql(
    """
    SELECT
        COUNT(CASE WHEN event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) AS purchases
    FROM ad_events;
    """, 
        engine)

display(metrics1)

,impressions,clicks,purchases
0,339812,40079,2031


In [8]:
#adding CTR and Conversion Rate
metrics2 = pd.read_sql(
    """
    SELECT
        COUNT(CASE WHEN event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) AS purchases,

        ROUND(
            COUNT(CASE WHEN event_type = 'Click' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN event_type = 'Impression' THEN 1 END),
            4
        ) AS ctr,

        ROUND(
            COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN event_type = 'Click' THEN 1 END),
            4
        ) AS conversion_rate
    FROM ad_events;
    """, 
        engine)

display(metrics2)

,impressions,clicks,purchases,ctr,conversion_rate
0,339812,40079,2031,0.1179,0.0507


***Insights***:
1. The **CTR of 11.79%** indicates that roughly 1 in 9 impressions results in a click, which reflects strong engagement at the top of the funnel.
2. The **conversion rate of 5.07%** means that about 5% of users who click proceed to purchase. This suggests that  there is noticeable drop-off between interest (click) and final action (purchase).

#### 2. Campaign Efficiency
This section answers:
* Which campaigns are efficient?
* Where is budget being used effectively?
* Which campaigns need optimization?

In [9]:
#campaign-level funnel metrics
cl_metrics = pd.read_sql(
    """
    SELECT
        c.campaign_id,
        c.name,
        c.total_budget,
        
        COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) AS purchases,
        
        ROUND(
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END),
            4
        ) AS ctr,

        ROUND(
            COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END),
            4
        ) AS conversion_rate
    FROM ad_events e
    JOIN ads a
        ON e.ad_id = a.ad_id
    JOIN campaigns c
        ON a.campaign_id = c.campaign_id

    GROUP BY c.campaign_id, c.name, c.total_budget
    ORDER BY purchases DESC;
    """, 
        engine)

display(cl_metrics)

,campaign_id,name,total_budget,impressions,clicks,purchases,ctr,conversion_rate
0,38,Campaign_38_Q3,71038.28,13425,1593,96,0.1187,0.0603
1,17,Campaign_17_Launch,86675.92,13506,1646,91,0.1219,0.0553
2,33,Campaign_33_Summer,59264.68,12043,1397,85,0.1160,0.0608
3,24,Campaign_24_Summer,56692.87,13638,1573,81,0.1153,0.0515
4,20,Campaign_20_Winter,98904.66,13607,1658,73,0.1218,0.0440
5,34,Campaign_34_Winter,26104.30,10205,1241,68,0.1216,0.0548
6,42,Campaign_42_Summer,7918.04,13671,1596,67,0.1167,0.0420
7,13,Campaign_13_Winter,21855.42,10331,1210,64,0.1171,0.0529
8,47,Campaign_47_Launch,69493.36,10276,1198,63,0.1166,0.0526
9,9,Campaign_9_Launch,40094.07,10343,1223,61,0.1182,0.0499


In [10]:
#cost per acquisition (cpa)
cpa = pd.read_sql(
    """
    SELECT
        c.campaign_id,
        c.name,
        c.total_budget,
        
        COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) AS purchases,
        
        ROUND(
            c.total_budget * 1.0 /
            NULLIF(COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END), 0),
            2
        ) AS cpa

    FROM ad_events e
    JOIN ads a
        ON e.ad_id = a.ad_id
    JOIN campaigns c
        ON a.campaign_id = c.campaign_id

    GROUP BY c.campaign_id, c.name, c.total_budget
    ORDER BY cpa DESC;
    """, 
        engine)

display(cpa)

,campaign_id,name,total_budget,purchases,cpa
0,35,Campaign_35_Launch,71626.83,11,6511.53
1,50,Campaign_50_Summer,47274.70,8,5909.34
2,36,Campaign_36_Q3,58801.99,10,5880.20
3,32,Campaign_32_Summer,81744.53,27,3027.58
4,45,Campaign_45_Summer,53303.55,19,2805.45
5,46,Campaign_46_Winter,94023.76,36,2611.77
6,2,Campaign_2_Launch,79342.41,31,2559.43
7,41,Campaign_41_Winter,85220.35,35,2434.87
8,6,Campaign_6_Winter,78607.49,33,2382.05
9,19,Campaign_19_Winter,33182.45,14,2370.18


***Interpretation***:
1. Some campaigns have high budgets but generate very few purchases, resulting in extremely high CPA.
2. Several **Winter campaigns** appear among the most efficient performers.
3. The best-performing campaigns (**CPA < 500**) deliver strong purchase volumes at relatively low budget levels.
4. The gap between worst and best CPA is substantial (**over 6,000 vs ~118**); it indicates clear optimization opportunities.


***Insight***:

Not all campaigns are equally efficient. Therefore, budget efficiency analysis is necessary for informed decision-making.

#### 3. Campaign Type Performance

In [11]:
#efficiency by campaign type
cpa = pd.read_sql(
    """
    WITH campaign_purchases AS (
        SELECT
            c.campaign_id,
            c.name,
            c.total_budget,

            COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) AS purchases
            
        FROM ad_events e
        JOIN ads a
            ON e.ad_id = a.ad_id
        JOIN campaigns c
            ON a.campaign_id = c.campaign_id

        GROUP BY c.campaign_id, c.name, c.total_budget
    )
    SELECT
        CASE
            WHEN name LIKE '%Winter%' THEN 'Winter'
            WHEN name LIKE '%Summer%' THEN 'Summer'
            WHEN name LIKE '%Launch%' THEN 'Launch'
            WHEN name LIKE '%Q3%' THEN 'Q3'
        END AS campaign_type,
        
        COUNT(*) AS campaign_count,
        SUM(total_budget) AS total_budget,
        SUM(purchases) AS total_purchases,
        
        ROUND(
            SUM(total_budget) * 1.0 /
            NULLIF(SUM(purchases), 0),
            2
        ) AS avg_cpa

    FROM campaign_purchases
    GROUP BY campaign_type
    ORDER BY avg_cpa ASC;
    """, 
        engine)

display(cpa)

,campaign_type,campaign_count,total_budget,total_purchases,avg_cpa
0,Winter,19,845546.53,793,1066.26
1,Summer,10,517519.66,438,1181.55
2,Q3,10,448502.38,377,1189.66
3,Launch,9,571483.03,423,1351.02


***Insights***:
1. Seasonal campaigns (especially Winter) appear to outperform Launch campaigns in terms of cost efficiency.
2. Launch campaigns may require optimization or strategic review.

**KEY OBSERVATION**:

Earlier, we noticed a huge cpa-variation at *individual campaign level* (CPA from ~118 to 6,500).

Now at *category level*, variation is much narrower (1,066 to 1,351).

This means that major efficiency differences happen within categories, not only between them.

#### 4. Platform Performance

In [12]:
#platform performance
platform_prf = pd.read_sql(
    """
    SELECT
        a.ad_platform,
        
        COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) AS purchases,

        ROUND(
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END),
            4
        ) AS ctr,

        ROUND(
            COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END),
            4
        ) AS conversion_rate
        
    FROM ad_events e
    JOIN ads a
        ON e.ad_id = a.ad_id

    GROUP BY a.ad_platform
    ORDER BY ctr DESC;
    """, 
        engine)

display(platform_prf)

,ad_platform,impressions,clicks,purchases,ctr,conversion_rate
0,Instagram,123840,14690,708,0.1186,0.0482
1,Facebook,215972,25389,1323,0.1176,0.0521


#### 5. Audience Performance

In [13]:
#audience performance
audience_prf = pd.read_sql(
    """
    SELECT
        u.user_gender,
        u.age_group,
        
        COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) AS purchases,

        ROUND(
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Impression' THEN 1 END),
            4
        ) AS ctr,

        ROUND(
            COUNT(CASE WHEN e.event_type = 'Purchase' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN e.event_type = 'Click' THEN 1 END),
            4
        ) AS conversion_rate
        
    FROM ad_events e
    JOIN users u
        ON e.user_id = u.user_id

    GROUP BY u.user_gender, u.age_group
    HAVING impressions > 5000
    ORDER BY purchases DESC;
    """, 
        engine)

display(audience_prf)

,user_gender,age_group,impressions,clicks,purchases,ctr,conversion_rate
0,Male,25-34,78590,9225,471,0.1174,0.0511
1,Male,18-24,59950,7056,363,0.1177,0.0514
2,Female,25-34,48448,5813,288,0.1200,0.0495
3,Female,18-24,36588,4293,223,0.1173,0.0519
4,Male,35-44,28049,3258,155,0.1162,0.0476
5,Female,35-44,16926,2047,126,0.1209,0.0616
6,Other,25-34,15210,1817,98,0.1195,0.0539
7,Male,16-17,16250,1891,94,0.1164,0.0497
8,Other,18-24,10206,1183,58,0.1159,0.0490
9,Female,16-17,10767,1240,53,0.1152,0.0427


***Insights***:
1. Both platforms perform similarly at the engagement level.
2. Facebook is slightly stronger in converting clicks into purchases.
3. Performance appears broadly stable across audience segments.
4. Smaller segments (e.g., “**Other 55-65**”) show higher volatility due to low impression counts.
5. To avoid misleading conclusions from low-volume segments, a minimum impression threshold was applied.

#### 6. Time-Based Performance Analysis

In this section, we want to answer something simple:
* Is performance improving over time?
* Do CTR and conversion fluctuate?

In [14]:
#daily trend
daily_trend = pd.read_sql(
    """
    SELECT
        STRFTIME('%Y-%W', timestamp) AS year_week,
        
        COUNT(CASE WHEN event_type = 'Impression' THEN 1 END) AS impressions,
        COUNT(CASE WHEN event_type = 'Click' THEN 1 END) AS clicks,
        COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) AS purchases,

        ROUND(
            COUNT(CASE WHEN event_type = 'Click' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN event_type = 'Impression' THEN 1 END),
            4
        ) AS ctr,

        ROUND(
            COUNT(CASE WHEN event_type = 'Purchase' THEN 1 END) * 1.0 /
            COUNT(CASE WHEN event_type = 'Click' THEN 1 END),
            4
        ) AS conversion_rate
        
    FROM ad_events e

    GROUP BY STRFTIME('%Y-%W', timestamp)
    ORDER BY year_week;
    """, 
        engine)

display(daily_trend)

,year_week,impressions,clicks,purchases,ctr,conversion_rate
0,2025-18,16589,2013,99,0.1213,0.0492
1,2025-19,26059,3101,148,0.1190,0.0477
2,2025-20,26308,3129,152,0.1189,0.0486
3,2025-21,26053,3112,161,0.1194,0.0517
4,2025-22,26310,3062,140,0.1164,0.0457
5,2025-23,26210,2978,156,0.1136,0.0524
6,2025-24,26204,2974,191,0.1135,0.0642
7,2025-25,26228,3026,150,0.1154,0.0496
8,2025-26,26094,3087,151,0.1183,0.0489
9,2025-27,25876,3046,153,0.1177,0.0502


***Insights***:
1. Performance is generally stable across weeks.
2. Engagement (CTR) does not fluctuate significantly.
3. Week 24 shows a noticeable improvement in conversion rate, it may require further investigation.

## Key Findings & Business Implications

#### 1. Funnel Performance

The overall funnel shows expected drop-off from impressions to clicks to purchases.
CTR is approximately **11.8%**, and conversion rate from click to purchase is around **5%**.

***Business Implication***:

* Improve post-click performance. Optimizing landing pages or purchase experience could increase final conversion.

#### 2. Campaign-Level Efficiency

There is substantial variation in CPA across individual campaigns; Some campaigns generate strong purchase volume at low cost, while others have high budget but low efficiency.

***Business Implication***:

* Reallocate budget toward high-performing campaigns could improve overall return.

#### 3. Campaign Type Performance

Winter campaigns show the lowest average CPA among campaign categories.
However, variation within campaign types is larger than variation between types.

***Business Implication***:

* Optimize individual campaigns rather than relying solely on campaign category.

#### 4. Platform Performance

Instagram and Facebook show similar engagement levels.
Facebook performs slightly better in post-click conversion.

***Business Implication***:

* Platform selection is not a major performance driver. Budget decisions should prioritize campaign efficiency rather than platform preference.

#### 5. Audience Performance

Performance metrics are relatively stable across major demographic groups.

***Business Implication***:

* Optimization efforts may be better directed toward campaign strategy and creative performance.

#### 6. Time-Based Trends

Weekly performance remains stable with consistent engagement and purchase levels. A minor spike in conversion was observed in one week, but no strong trend is present.

***Business Implication***:

* Campaign performance appears steady over time.